# aiohttp smoke test

Client and server are exercised against each other on `127.0.0.1` with an
ephemeral port, so the notebook runs offline.

In [ ]:
# Example 1: routing + POST JSON round-trip.
import aiohttp
from aiohttp import web

async def echo_json(request):
    data = await request.json()
    return web.json_response({"received": data, "processed": True})

app = web.Application()
app.router.add_post("/data", echo_json)
runner = web.AppRunner(app)
await runner.setup()
site = web.TCPSite(runner, "127.0.0.1", 0)  # port 0 = ephemeral, parallel-safe
await site.start()
port = runner.addresses[0][1]

payload = {"kind": "smoke", "n": 3}
async with aiohttp.ClientSession() as session:
    async with session.post(f"http://127.0.0.1:{port}/data", json=payload) as resp:
        body = await resp.json()
assert body == {"received": payload, "processed": True}
print(f"POST round-trip body: {body}")
await runner.cleanup()
print("Example 1: routing + JSON POST round-trip OK")


In [ ]:
# Example 2: middleware — errors converted to JSON 500s on a real request.
import aiohttp
from aiohttp import web

@web.middleware
async def error_middleware(request, handler):
    try:
        return await handler(request)
    except web.HTTPException:
        raise  # let real HTTP errors pass through
    except Exception as e:
        return web.json_response({"error": "Internal server error", "message": str(e)}, status=500)

async def boom(request):
    raise ValueError("intentional failure")

app = web.Application(middlewares=[error_middleware])
app.router.add_get("/boom", boom)
runner = web.AppRunner(app)
await runner.setup()
site = web.TCPSite(runner, "127.0.0.1", 0)  # port 0 = ephemeral, parallel-safe
await site.start()
port = runner.addresses[0][1]

async with aiohttp.ClientSession() as session:
    async with session.get(f"http://127.0.0.1:{port}/boom") as resp:
        body = await resp.json()
        assert resp.status == 500 and body["message"] == "intentional failure"
        print(f"middleware converted exception -> {resp.status}: {body}")
await runner.cleanup()
print("Example 2: middleware error handling OK")


In [ ]:
# Example 3: cookie round-trip — server set_cookie -> client CookieJar -> server read.
import aiohttp
from aiohttp import web

async def set_cookie(request):
    resp = web.Response(text="cookie set")
    resp.set_cookie("user_id", "user123", max_age=3600)
    return resp

async def read_cookie(request):
    return web.json_response({"user_id": request.cookies.get("user_id", "anonymous")})

app = web.Application()
app.router.add_get("/set", set_cookie)
app.router.add_get("/read", read_cookie)
runner = web.AppRunner(app)
await runner.setup()
site = web.TCPSite(runner, "127.0.0.1", 0)  # port 0 = ephemeral, parallel-safe
await site.start()
port = runner.addresses[0][1]

jar = aiohttp.CookieJar(unsafe=True)  # unsafe=True: accept cookies from bare-IP hosts
async with aiohttp.ClientSession(cookie_jar=jar) as session:
    async with session.get(f"http://127.0.0.1:{port}/set") as resp:
        await resp.text()
    print("cookies in jar:", {c.key: c.value for c in jar})
    async with session.get(f"http://127.0.0.1:{port}/read") as resp:
        body = await resp.json()
assert body == {"user_id": "user123"}
await runner.cleanup()
print("Example 3: cookie round-trip OK")


In [ ]:
# Example 4: WebSocket echo round-trip (ws_connect client <-> WebSocketResponse server).
import json
import aiohttp
from aiohttp import web, WSMsgType

async def ws_handler(request):
    ws = web.WebSocketResponse()
    await ws.prepare(request)
    async for msg in ws:
        if msg.type == WSMsgType.TEXT:
            await ws.send_str(json.dumps({"echo": json.loads(msg.data)["message"]}))
    return ws

app = web.Application()
app.router.add_get("/ws", ws_handler)
runner = web.AppRunner(app)
await runner.setup()
site = web.TCPSite(runner, "127.0.0.1", 0)  # port 0 = ephemeral, parallel-safe
await site.start()
port = runner.addresses[0][1]

async with aiohttp.ClientSession() as session:
    async with session.ws_connect(f"http://127.0.0.1:{port}/ws") as ws:
        await ws.send_str(json.dumps({"message": "ping"}))
        reply = json.loads((await ws.receive()).data)
        await ws.close()
assert reply == {"echo": "ping"}
await runner.cleanup()
print(f"Example 4: WebSocket echo round-trip OK: {reply}")


In [ ]:
# Example 5: request/headers round-trip — the handler sees the method, path, a custom header and the body.
import aiohttp
from aiohttp import web

seen = {}


async def echo_request(request):
    seen["message_module"] = type(request.message).__module__  # informational only
    body = await request.read()
    return web.json_response(
        {"method": request.method, "path": request.path, "x_probe": request.headers["X-Probe"], "body_len": len(body)}
    )


app = web.Application()
app.router.add_post("/echo", echo_request)
runner = web.AppRunner(app)
await runner.setup()
site = web.TCPSite(runner, "127.0.0.1", 0)  # port 0 = ephemeral, parallel-safe
await site.start()
port = runner.addresses[0][1]
try:
    async with aiohttp.ClientSession() as session:
        async with session.post(f"http://127.0.0.1:{port}/echo", data=b"x" * 1024, headers={"X-Probe": "smoke"}) as resp:
            assert resp.status == 200, resp.status
            body = await resp.json()
finally:
    await runner.cleanup()
assert body == {"method": "POST", "path": "/echo", "x_probe": "smoke", "body_len": 1024}, body
print(f"Example 5: request/headers round-trip OK (request message type from {seen['message_module']})")